In [0]:
%pip install -U \
  langchain \
  langchain-core \
  langchain-text-splitters

dbutils.library.restartPython()


In [0]:
# Cell 1 — Setup
from databricks.sdk import WorkspaceClient

# Cell 1 — add EndpointType to your import
from databricks.sdk.service.vectorsearch import (
    VectorIndexType, DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn, PipelineType, EndpointType
)

w        = WorkspaceClient()
username = w.current_user.me().user_name.split("@")[0].replace(".", "_").replace("-", "_")
catalog  = "bootcamp_students"
schema   = f"maintops"
VS_ENDPOINT = "zachy_vs"
print(f"Working in: {catalog}.{schema}")

In [0]:
# Create chunked_bootcamp_docs table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.chunked_faq (
  chunk_id     STRING    COMMENT 'Unique chunk identifier (UUID) — primary key for Vector Search',
  doc_id       STRING    COMMENT 'Foreign key to raw_faq.doc_id',
  chunk_index  INT       COMMENT 'Zero-based position of this chunk within the source document',
  chunk_text   STRING    COMMENT 'Chunk plain text — the field that will be embedded',
  source_type  STRING    COMMENT 'Inherited from raw_maintops_docs: pdf | web | image',
  source_url   STRING    COMMENT 'Provenance URL or file path',
  title        STRING    COMMENT 'Source document title',
  chunked_at   TIMESTAMP COMMENT 'Timestamp when the chunk was created'
)
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print(f"Table ready: {catalog}.{schema}.chunked_faq")
display(spark.sql(f"DESCRIBE TABLE {catalog}.{schema}.chunked_faq"))

In [0]:
# Chunk raw_maintops_docs and write to chunked_bootcamp_docs
import uuid
from datetime import datetime, timezone
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql import Row

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

# Load all source documents (skip rows with no content)
source_docs = spark.table(f"{catalog}.{schema}.raw_faq") \
    .filter("content IS NOT NULL AND length(trim(content)) > 0") \
    .select("doc_id", "source_type", "source_url", "title", "content") \
    .collect() 
    
### Note: this is loading all records to the driver node as a list of Row objects. This is fine for our small dataset, but for larger datasets, consider processing in batches or using Spark UDFs.

chunked_rows = []
for doc in source_docs:
    chunks = recursive_splitter.create_documents(
        [doc.content],
        metadatas=[{"doc_id": doc.doc_id}],
    )
    for i, chunk in enumerate(chunks):
        chunked_rows.append(Row(
            chunk_id    = str(uuid.uuid4()),
            doc_id      = doc.doc_id,
            chunk_index = i,
            chunk_text  = chunk.page_content,
            source_type = doc.source_type,
            source_url  = doc.source_url,
            title       = doc.title,
            chunked_at  = datetime.now(timezone.utc),
        ))

print(f"Source documents: {len(source_docs)}")
print(f"Total chunks produced: {len(chunked_rows)}")

# Write (overwrite to make this cell idempotent)
spark.createDataFrame(chunked_rows) \
    .write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{schema}.chunked_faq")

display(spark.table(f"{catalog}.{schema}.chunked_faq").limit(10))

In [0]:
# Cell 2 — Create VS endpoint (shared across the bootcamp workspace)
# Endpoint types:
#   - STANDARD (default): recommended for most workloads
#   - STORAGE_OPTIMIZED: for large indexes (>1B vectors), lower cost
try:
    w.vector_search_endpoints.create_endpoint(
        name=VS_ENDPOINT,
        endpoint_type=EndpointType.STANDARD   # Use STORAGE_OPTIMIZED for very large indexes
    ).result()
    
    print(f"Endpoint created: {VS_ENDPOINT} (STANDARD)")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint already exists: {VS_ENDPOINT} — reusing.")
    else:
        raise

In [0]:
# Cell 3 — Create the bootcamp docs index via raw REST API
index_name = f"{catalog}.{schema}.faq_index"

try:
    response = w.api_client.do(
        "POST",
        "/api/2.0/vector-search/indexes",
        body={
            "name": index_name,
            "endpoint_name": VS_ENDPOINT,
            "primary_key": "chunk_id",
            "index_type": "DELTA_SYNC",
            "delta_sync_index_spec": {
                "source_table": f"{catalog}.{schema}.chunked_faq",
                "pipeline_type": "TRIGGERED",
                "embedding_source_columns": [
                    {
                        "name": "chunk_text",
                        "embedding_model_endpoint_name": "databricks-gte-large-en"
                    }
                ]
            }
        }
    )
    print(f"Index ready: {response['name']}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Index already exists — skipping creation.")
    else:
        raise

In [0]:
# Cell 3b — Wait for the index's initial sync to finish before querying it.
# Index creation above uses PipelineType.TRIGGERED, which is asynchronous — querying
# immediately after creation is a race condition (the index may not be ONLINE/ready yet).
# A brand-new endpoint's first sync can take longer than a warm re-sync, hence the 900s budget.
import time

def wait_for_index_ready(index_name: str, timeout_s: int = 900, poll_s: int = 10):
    start = time.time()
    while time.time() - start < timeout_s:
        status = w.api_client.do("GET", f"/api/2.0/vector-search/indexes/{index_name}")
        state = status.get("status", {}).get("detailed_state") or status.get("status", {}).get("state")
        ready = status.get("status", {}).get("ready", False)
        print(f"  index state: {state}  ready={ready}  ({int(time.time() - start)}s elapsed)")
        if ready:
            return status
        time.sleep(poll_s)
    raise TimeoutError(f"Index {index_name} did not become ready within {timeout_s}s")

print("Waiting for initial index sync...")
wait_for_index_ready(index_name)
print("Index is ready to query.")

In [0]:
# Cell 4 — Query the index (dense/ANN retrieval)
# Note: Hybrid is now recommended as the default. Dense is better for pure semantic queries.
results = w.api_client.do("POST",
    f"/api/2.0/vector-search/indexes/{index_name}/query",
    body={
        "columns": ["chunk_id", "chunk_text"],
        "query_text": "How does the recommendation process work?",
        "num_results": 5,
        "query_type": "hybrid"  # Dense only (baseline for comparison)
    }
)

print("Top 5 results (Hybrid retrieval):")
for i, row in enumerate(results["result"]["data_array"], 1):
    print(f"\n[{i}] {row[2]}")
    print(f"    {str(row[1])[:200]}...")